In [1]:
import sys
sys.path.append("..")
import os
import torch
import json
import yaml
from pathlib import Path
from datasets import DatasetDict, load_from_disk
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

from transformers import AutoTokenizer, AutoModelForCausalLM
from src.data import PROJECT_ROOT
from src.dataset_cot import prepare_direct_pythia_dataset, get_cot_prompt_template, evaluate_cot_capability
from src.llm_upgrade import (
    prepare_model_for_finetune,
    train_lora_model,
    save_finetuned_model,
    wrap_for_transformer_lens,
    create_quantization_config
)

# Параметры (q)LoRA

In [2]:
EXP_ID = "exp6-2"
MODEL_NAME = "gpt2-large"
VARIANT = "depth-1"
USE_SMALL = False

In [3]:
EPOCHS = 4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1.0e-4
MAX_LENGTH = 512
SAVE_STEPS = 1000
LOGGING_STEPS = 200
EVAL_STEPS = 200

In [4]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

In [5]:
TARGET_MODULES = ["c_attn", "c_proj", "c_fc"]

In [6]:
OUTPUT_DIR = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}")
ADAPTER_SAVE_PATH = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/lora_final")

In [7]:
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
raw_dataset = load_from_disk(str(cache_path))

In [11]:
# TRAIN_SIZE = 10000
DEV_SIZE = 1000
SEED = 42

# Подготовка датасета

In [16]:
train_subset = raw_dataset["train"].shuffle(seed=SEED)
dev_subset = raw_dataset["dev"].shuffle(seed=SEED).select(range(min(DEV_SIZE, len(raw_dataset["dev"]))))

In [17]:
train_dataset = prepare_direct_pythia_dataset(train_subset)
dev_dataset   = prepare_direct_pythia_dataset(dev_subset)

In [18]:
len(train_dataset)

61699

In [19]:
len(dev_subset)

1000

In [20]:
direct_dir = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}_direct_gpt2"
if dev_dataset:
    DatasetDict({"train": train_dataset, "dev": dev_dataset, "test": raw_dataset["test"]}).save_to_disk(str(direct_dir))

Saving the dataset (0/1 shards):   0%|          | 0/61699 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17788 [00:00<?, ? examples/s]

# Обучение

In [21]:
# qlora
quantization_config = create_quantization_config(use_4bit=True)

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [23]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"  # Flash Attention часто нестабилен с LoRA на Windows
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [24]:
base_model = prepare_model_for_kbit_training(base_model)

In [25]:
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False
)

In [26]:
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

trainable params: 11,796,480 || all params: 785,826,560 || trainable%: 1.5012


In [27]:
train_config = {
    "output_dir": OUTPUT_DIR,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "logging_steps": LOGGING_STEPS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "use_wandb": False
}

In [28]:
trained_model, metrics = train_lora_model(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    config=train_config,
    max_length=MAX_LENGTH
)

Map:   0%|          | 0/61699 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,1.668700,0.926775
400,0.849500,0.794759
600,0.784100,0.774231
800,0.757900,0.758113
1000,0.749700,0.750617
1200,0.738400,0.746537
1400,0.727800,0.747640
1600,0.731100,0.745826
1800,0.725600,0.741110
2000,0.722300,0.733604


c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\MyPythonProjects\meph

In [29]:
save_finetuned_model(trained_model, tokenizer, ADAPTER_SAVE_PATH)

Адаптер сохранён в C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp6-2\lora_final


# Тест дообученной модели

In [30]:
BEST_CHECKPOINT = 5000
adapter_path = PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/checkpoint-{BEST_CHECKPOINT}"

In [31]:
hooked_model, _ = wrap_for_transformer_lens(
    MODEL_NAME,
    str(adapter_path),
    device="cuda"
)

c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Loaded pretrained model gpt2-large into HookedTransformer


In [32]:
eval_prefix = "{theory} {assertion} The assertion is"

In [33]:
full_test = load_from_disk(str(cache_path))["test"]

In [34]:
test_acc = evaluate_cot_capability(
    model=hooked_model,
    dataset=full_test,
    prompt_template=eval_prefix,
    batch_size=32,
    device="cuda"
)

Evaluating CoT capability: 100%|██████████| 556/556 [1:34:45<00:00, 10.22s/it]


In [35]:
results = {
    "experiment_id": EXP_ID,
    "model": MODEL_NAME,
    "variant": VARIANT,
    "best_checkpoint": f"checkpoint-{BEST_CHECKPOINT}",
    "val_loss_at_checkpoint": 0.718434,
    "test_accuracy": float(test_acc),
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lora_r": LORA_R, "train_size": "full"
    }
}

In [36]:
results_path = Path(OUTPUT_DIR) / "results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

In [37]:
print(f"Experiment: {EXP_ID}")
print(f"Checkpoint: {adapter_path.name}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Results saved: {results_path}")

Experiment: exp6-2
Checkpoint: checkpoint-5000
Test Accuracy: 0.5022
Results saved: C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp6-2\results.json


# Диагностика предсказаний

In [38]:
test_examples = raw_dataset["test"].select(range(5))
eval_prefix = "{theory} {assertion} The assertion is"

In [39]:
with torch.no_grad():
    for i, ex in enumerate(test_examples):
        prompt = eval_prefix.format(theory=ex["theory"], assertion=ex["assertion"])
        tokens = hooked_model.to_tokens([prompt], prepend_bos=True).to("cuda")
        logits = hooked_model(tokens)

        # Индексы целевых токенов
        true_id = hooked_model.tokenizer.encode("True", add_special_tokens=False)[-1]
        false_id = hooked_model.tokenizer.encode("False", add_special_tokens=False)[-1]

        # Длина промпта (последний токен находится по этому индексу после добавления BOS)
        prompt_len = len(hooked_model.tokenizer.encode(prompt, add_special_tokens=False))

        logit_t = logits[0, prompt_len, true_id].item()
        logit_f = logits[0, prompt_len, false_id].item()
        pred = "True" if logit_t > logit_f else "False"
        true_label = "True" if ex["label"] else "False"

        print(f"[{i+1}] Предсказание: {pred:5s} | Логиты T/F: {logit_t:+.2f} / {logit_f:+.2f} | Истина: {true_label} | Совпало: {pred == true_label}")

[1] Предсказание: True  | Логиты T/F: +10.00 / +9.12 | Истина: True | Совпало: True
[2] Предсказание: True  | Логиты T/F: +10.12 / +9.00 | Истина: False | Совпало: False
[3] Предсказание: True  | Логиты T/F: +10.12 / +9.19 | Истина: True | Совпало: True
[4] Предсказание: True  | Логиты T/F: +10.12 / +9.00 | Истина: False | Совпало: False
[5] Предсказание: True  | Логиты T/F: +10.25 / +9.06 | Истина: True | Совпало: True


In [40]:
del hooked_model
torch.cuda.empty_cache()